# NPTEL Study Notes: Decoding Strategies in Sequence-to-Sequence Models

## 1. Introduction to Decoding
In sequence-to-sequence models (like Encoder-Decoder LSTMs or Transformers), the model generates output sequences token by token. Examples include Machine Translation (e.g., English to Hindi) or Question Answering.

At each time step $t$, the model produces a probability distribution over the entire vocabulary $V$. The core problem of **decoding** is choosing the specific next token $w_t$ from this distribution.

**Mathematical Context:**
The model calculates the probability of the next word given the initial context/prompt and all previously generated words:
$$P(w_t | C, W_{<t})$$
Where:
*   $C$ = The input sequence or prompt (e.g., "I am going to school").
*   $W_{<t}$ = The tokens generated so far up to time step $t-1$.
*   $w_t$ = The current token to be generated at time step $t$.

### The Core Trade-off: Quality vs. Diversity
*   **Quality (Deterministic):** Generating the absolute most probable response. Best for factual tasks like Question Answering or Machine Translation, where correctness matters more than creativity.
*   **Diversity (Stochastic):** Generating varied, creative responses. Best for tasks like story generation or writing a poem, where you do not want the exact same output every time.

---

## 2. Deterministic Decoding Strategies
These strategies do not involve randomness. Given the same input, they will always produce the exact same output.

### 2.1 Greedy Decoding
*   **Mechanism:** At every time step, simply choose the single most probable token from the distribution.
*   **Advantage:** Fast, computationally cheap, and generally produces accurate, high-quality factual responses.
*   **Disadvantage:** Lacks diversity. It will deterministically produce the exact same sequence every time. It can also get trapped in suboptimal sequences because it only looks one step ahead.
*   **Example:** If $P(\text{cheese}) = 0.35$ and $P(\text{toppings}) = 0.20$, Greedy Decoding will *always* choose "cheese".

### 2.2 Exhaustive Decoding
*   **Mechanism:** Explore all possible sequences of length $N$ over the entire vocabulary $V$.
*   **Complexity:** For a vocabulary size $|V|$, there are $|V|^N$ possible sequences.
*   **Verdict:** Computationally impossible in practice due to exponential time complexity.

### 2.3 Beam Search
*   **Mechanism:** A practical compromise to exhaustive search. Instead of just keeping the single best word (Greedy) or all words (Exhaustive), Beam Search keeps track of the $K$ most probable sequences (the "beam size") at each time step.
*   **Process:** At each step, it expands the $K$ current sequences, calculates probabilities, and prunes the branches that fall outside the top $K$ cumulative probabilities.
*   **Verdict:** Still deterministic, but yields better global sequence quality than Greedy Decoding.

---

## 3. Stochastic (Probabilistic) Decoding Strategies
To introduce creativity and diversity, we use stochastic strategies that sample from the probability distribution rather than just picking the maximum value.

### 3.1 Random Sampling
*   **Mechanism:** Randomly sample a word from the entire vocabulary distribution.
*   **Verdict:** Usually a bad strategy. It gives a non-zero chance to highly irrelevant or grammatically incorrect words, leading to poor-quality text.

### 3.2 Top-K Sampling
*   **Mechanism:** 
    1. Sort the probability distribution and select the top $K$ highly probable tokens.
    2. **Renormalize** the probabilities of these $K$ tokens so they sum to 1.
    3. Randomly sample the next word from this restricted, renormalized set.
*   **Example:** Let $K=3$. 
    *   *Original Top 3:* Cheese (0.35), Toppings (0.20), Pepperoni (0.12).
    *   *Renormalization sum:* $0.35 + 0.20 + 0.12 = 0.67$.
    *   *Renormalized $P(\text{cheese})$:* $0.35 / 0.67 \approx 0.52$.
    *   Sample randomly using these new probabilities.
*   **Disadvantage (The Skewness Problem):** Fixing $K$ is problematic because distributions change shape. 
    *   *Skewed Distribution:* Top 2 words might hold 99% of the probability mass. A fixed $K=10$ forces the model to include 8 highly unlikely/irrelevant words in the sampling pool.
    *   *Uniform Distribution:* Top 10 words might be equally good. A fixed $K=3$ unfairly cuts off 7 perfectly valid choices.

### 3.3 Top-p (Nucleus) Sampling
*   **Mechanism:** Instead of fixing the *number* of tokens ($K$), we fix a cumulative probability threshold $p$ (e.g., $p = 0.5$ or $0.6$). 
    1. Sort tokens by probability in descending order.
    2. Keep adding tokens to the candidate set until the sum of their probabilities $\geq p$.
    3. Renormalize this subset and sample.
*   **Advantage over Top-K:** It dynamically adjusts the candidate set size based on the shape of the distribution.
    *   If the distribution is *skewed*, it might only pick 1-2 tokens to reach $p = 0.5$.
    *   If the distribution is *uniform*, it might pick 10 tokens to reach $p = 0.5$.

### 3.4 Temperature Sampling
*   **Mechanism:** Rather than truncating the distribution (like Top-K or Top-p), Temperature Sampling modifies the **Softmax** function used to convert raw model scores (logits, $z_i$) into probabilities.
*   **Formula:** 
    $$p_i = \frac{\exp(z_i / \tau)}{\sum_j \exp(z_j / \tau)}$$
    Where $\tau$ is the temperature hyperparameter.
*   **Impact of $\tau$:**
    *   **$\tau = 1$:** Standard Softmax. Unchanged probabilities.
    *   **$\tau < 1$ (Low Temperature):** Decreases the denominator in the exponent, magnifying differences between logits. The distribution becomes highly **skewed** (sharper peaks). Makes the output more **deterministic** and focused on the highest probability tokens.
    *   **$\tau > 1$ (High Temperature):** Flattens the distribution towards a uniform shape. Gives lower-probability words a higher chance of being selected. Promotes **creativity, diversity, and stochasticity**.

---

## 4. NPTEL Practice Questions

**Q1. Which of the following decoding strategies dynamically adjusts the number of candidate tokens based on the shape of the probability distribution?**
A) Greedy Decoding
B) Beam Search
C) Top-K Sampling
D) Top-p (Nucleus) Sampling
*   **Answer:** D
*   **Explanation:** Top-p sampling selects the smallest set of tokens whose cumulative probability exceeds a threshold $p$. If the distribution is flat, it selects more tokens; if skewed, fewer. Top-K fixes the number of tokens regardless of the distribution.

**Q2. You are designing an LLM application to write creative poetry. You are using Temperature Sampling. How should you adjust the temperature hyperparameter ($\tau$)?**
A) Set $\tau$ to a value close to 0 to ensure high accuracy.
B) Set $\tau$ to a high value (e.g., $>1$) to flatten the distribution and promote diversity.
C) Keep $\tau$ exactly at 1 to maintain deterministic behavior.
D) Temperature sampling is deterministic, so the value of $\tau$ does not matter.
*   **Answer:** B
*   **Explanation:** A higher temperature ($\tau$) makes the softmax distribution more uniform, allowing the model to sample less probable, more creative tokens, which is ideal for poetry. Lower values make the model more deterministic.

**Q3. A sequence-to-sequence model produces the following probabilities for the next token: W1: 0.45, W2: 0.30, W3: 0.15, W4: 0.10. If Top-p sampling is applied with a threshold $p = 0.7$, which tokens will form the candidate set before renormalization?**
A) W1 only
B) W1 and W2
C) W1, W2, and W3
D) All tokens
*   **Answer:** B
*   **Explanation:** We sort descending (already sorted). We add probabilities until we cross $0.7$. W1 (0.45) is not enough. W1 + W2 (0.45 + 0.30 = 0.75). Since 0.75 $\geq$ 0.7, the threshold is met, and the smallest set is {W1, W2}.

---

## 5. Quick Revision Summary
*   **Decoding:** Choosing the next token based on $P(w_t | C, W_{<t})$.
*   **Deterministic vs. Stochastic:** Deterministic methods (Greedy, Beam Search) are good for quality/accuracy but lack diversity. Stochastic methods (Sampling) enable creativity.
*   **Greedy vs. Beam:** Greedy picks the single best word. Beam tracks $K$ best sequences. Both are deterministic.
*   **Top-K Sampling:** Samples from the $K$ most probable words. Struggles with shifting distribution shapes.
*   **Top-p (Nucleus) Sampling:** Samples from the minimum number of tokens forming cumulative probability $p$. Adapts well to skewed vs. uniform distributions.
*   **Temperature ($\tau$):** Modifies Softmax. $\tau < 1$ = sharp, deterministic, quality-focused. $\tau > 1$ = flat, random, creative.